# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant JSON-LD schema available at this URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and explore high-level information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else ''}")

## 2. Data Overview
Explore available record sets and their corresponding field `@id`s.

If the schema includes multiple record sets, list their `@id`s, names, and the fields they contain.

In [ ]:
# List all record sets (@id, name) and their field @ids/names
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '[No Name]')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for fld in fields:
            # If only a @id provided, show
            if isinstance(fld, dict):
                print(f"    - {fld.get('@id', str(fld))}")
            else:
                print(f"    - {str(fld)}")
        print()

## 3. Data Extraction
Load data from each available record set into DataFrames for further analysis.
Reference all record set and field/column entities using their `@id`.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs["@id"] for rs in record_sets] if record_sets else []

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet: {record_set_id} (shape: {df.shape})")
    print(f"Columns: {df.columns.tolist()}\n")

# If any dataframes loaded, display first 5 rows of the first one
if dataframes:
    first_rs = record_set_ids[0]
    print(f"First records from RecordSet '{first_rs}':")
    display(dataframes[first_rs].head())
else:
    print("No record sets with data to load.")

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA steps: filtering, normalization, grouping. Use example numeric and grouping field `@id`s; adjust as needed to actual field ids from above.

In [ ]:
# For demonstration, attempt EDA on the first available record set/field
import numpy as np

if dataframes:
    # Choose first record set
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Attempt to find a numeric field by dtype
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dropna().apply(type), np.number).any() or pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field is not None:
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        try:
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())

            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try using a categorical/group field
            group_field = None
            for col in df.columns:
                if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                    group_field = col
                    break
            if group_field:
                print(f"Grouped data by '{group_field}':")
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                display(grouped_df.head())
            else:
                print("No suitable categorical/grouping field found.")
        except Exception as e:
            print(f"Unable to filter or normalize numeric field due to: {e}")
    else:
        print("No numeric fields available for analysis in the first record set.")
else:
    print("No data to perform EDA on.")

## 5. Visualization
Visualize field distributions, relationships, or summary statistics for numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_cols:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_cols[0]].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_cols[0]} (@id: {numeric_cols[0]})")
        plt.xlabel(numeric_cols[0])
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric columns to plot.")

    # Try categorical plot if possible
    cat_cols = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
    if cat_cols and numeric_cols:
        plt.figure(figsize=(8,4))
        # Barplot of mean value by first categorical column
        sns.barplot(data=df, x=cat_cols[0], y=numeric_cols[0], ci=None)
        plt.title(f"Mean {numeric_cols[0]} by {cat_cols[0]}")
        plt.xlabel(cat_cols[0])
        plt.ylabel(f"Mean {numeric_cols[0]}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data to visualize.")

## 6. Conclusion
This notebook demonstrated loading, overview, and basic processing of the FAIR² dataset using `mlcroissant` referencing all data elements by their `@id`. For in-depth analysis, further domain familiarization and code adaptation to specific schema field IDs are recommended.